###end to end pipeline


In [0]:
#we have users and products 

In [0]:
from pyspark.sql import SparkSession
spark = SparkSession.builder\
.appName("pyspark_optimization")\
.getOrCreate()

spark

In [0]:
from pyspark.sql.functions import rand

In [0]:
orders = spark.range(0, 5_000_000).withColumnRenamed("id", "order_id") \
    .withColumn("user_id", (rand()*100000).cast("int")) \
    .withColumn("product_id", (rand()*1000).cast("int")) \
    .withColumn("category", (rand()*10).cast("int")) \
    .withColumn("amount", rand()*1000)
 

In [0]:
#write orders

#data injestion layer

orders.write.mode("overwrite").parquet("/Volumes/workspace/default/pipeline/orders")

In [0]:
users = spark.range(0, 100000).withColumnRenamed("id", "user_id") \
    .withColumn("country", (rand()*5).cast("int"))

In [0]:
users.write.mode("overwrite").parquet("/Volumes/workspace/default/pipeline/users")

In [0]:
products = spark.range(0, 1000).withColumnRenamed("id", "product_id") \
    .withColumn("price", rand()*500)

In [0]:
products.write.mode("overwrite").parquet("/Volumes/workspace/default/pipeline/products")



In [0]:
#read the data
orders = spark. read.parquet("/Volumes/workspace/default/pipeline/orders")
users = spark. read. parquet ("/Volumes/workspace/default/pipeline/users")
products = spark. read. parquet("/Volumes/workspace/default/pipeline/products")

In [0]:
users.printSchema()

root
 |-- user_id: long (nullable = true)
 |-- country: integer (nullable = true)



In [0]:
orders.show(5)

+--------+-------+----------+--------+------------------+
|order_id|user_id|product_id|category|            amount|
+--------+-------+----------+--------+------------------+
| 1875000|  71994|       537|       9| 324.3920578483439|
| 1875001|  49670|       977|       6|  808.698298835842|
| 1875002|  74476|       974|       1|10.044578432862995|
| 1875003|  34875|       206|       7|146.77384530378623|
| 1875004|  61458|       522|       2|  960.437021373279|
+--------+-------+----------+--------+------------------+
only showing top 5 rows


In [0]:
#Transformation
#step 1. cleaning the category


from pyspark. sql.functions import col, lower, trim , broadcast, sum as _sum


In [0]:
orders_clean1 = orders.withColumn(
                                  "category", 
                                  col("category").cast("string")
                                  )

In [0]:
orders_clean= orders_clean1.withColumn(
    "category" , 
    lower(trim(col("category")))
) 

In [0]:
#Re-partition
orders_clean=orders_clean.repartition("user_id")
#it will redistribute the data across the clustre based on user id so that processing is more balanced.
#repartition is wide transformation.

In [0]:
#optimizing joins 
#using broadcast
df=orders_clean.join(broadcast(users), "user_id") \
    .join(products , "product_id")

caching 
df.cache()

In [0]:
#find top product & revenue  
top_products = df.groupBy("product_id")\
    .count()\
    .orderBy(col("count").desc())\
    .limit(10)

#this is top n aggregation process
#count - wide trans.

In [0]:
top_products.show(10)

+----------+-----+
|product_id|count|
+----------+-----+
|       768| 5262|
|       718| 5245|
|       882| 5214|
|       413| 5209|
|       399| 5189|
|       867| 5188|
|         3| 5187|
|       925| 5187|
|       641| 5173|
|        65| 5173|
+----------+-----+



In [0]:
#revenue per country
revenue = df.groupBy("country")\
    .agg(_sum("amount").alias("total_revenue"))


In [0]:
#write data

top_products.write.mode("overwrite").parquet("/Volumes/workspace/default/pipeline/output/top_products")

In [0]:
revenue.write.mode("overwrite").parquet("/Volumes/workspace/default/pipeline/output/revenue")